In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if (root / "src").exists():
    sys.path.append(str(root / "src"))
elif (root.parent / "src").exists():
    sys.path.append(str(root.parent / "src"))

from collections import Counter
from dataset import load_dataset, count_samples, get_label_mapping
import numpy as np

X, y = load_dataset()
sample_count = count_samples()
label_mapping = get_label_mapping()
number_of_classes = len(label_mapping)
files_per_class = {name: Counter(y)[idx] for name, idx in label_mapping.items()}
_first = np.load(X[0], allow_pickle=True)
first_sample_shape = _first.shape
first_sample_dtype = _first.dtype

def format_files_per_class(items):
    return {name: items[name] for name in sorted(items)}

print("Number of classes:", number_of_classes)
print("Number of files per class:", format_files_per_class(files_per_class))
print("Shape of the first sample:", first_sample_shape)
print("Data type:", first_sample_dtype)
print("Total number of samples:", sample_count)

Number of classes: 4
Number of files per class: {'ADS_B': 160, 'FM_broadcast': 160, 'ISM_sensors': 160, 'noise': 160}
Shape of the first sample: ()
Data type: object
Total number of samples: 640


In [2]:
from model_1d.preprocess import summarize_dataset

summary = summarize_dataset(X, y, batch_size=32, hop_length=2048)

print(f"Loaded {summary['loaded_samples']} samples")
print(f"\nOriginal sample shape: {summary['original_sample_shape']}")
print(f"Shape after preprocessing: {summary['preprocessed_sample_shape']}")
print(f"Data type: {summary['data_type']}")
print(f"\nTrain: {summary['train']}")
print(f"Validation: {summary['validation']}")
print(f"Test: {summary['test']}")
print(f"\nBatch shape: {summary['batch_shape']}")
print("\nTraining class distribution:")
for label, count in summary["train_class_distribution"].items():
    print(f"Label {label}: {count} samples")

Loaded 640 samples

Original sample shape: (512000,)
Shape after preprocessing: (2048, 2)
Data type: torch.float32

Train: 110900
Validation: 16167
Test: 31443

Batch shape: torch.Size([32, 2048, 2])

Training class distribution:
Label 0: 28328 samples
Label 1: 26593 samples
Label 2: 29043 samples
Label 3: 26936 samples


In [3]:
import torch

from model_1d.model import SignalCNN


def count_trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SignalCNN(num_classes=4).to(device)
dummy_input = torch.randn(1, 2, 2048, device=device)
output = model(dummy_input)
predicted_class = torch.argmax(output, dim=1)

print("Model architecture:")
print(model)
print()
print("Number of trainable parameters:", count_trainable_parameters(model))
print("Device:", device)
print("Dummy input shape:", tuple(dummy_input.shape))
print("Output shape:", tuple(output.shape))
print("Output values:")
print(output)
print("Predicted class:", predicted_class.item())
print("Any NaNs in input:", torch.isnan(dummy_input).any().item())
print("Any NaNs in output:", torch.isnan(output).any().item())

Model architecture:
SignalCNN(
  (features): Sequential(
    (0): Conv1d(2, 32, kernel_size=(9,), stride=(1,), padding=(4,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (12): AdaptiveAvgPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.3

In [4]:
from model_1d.train import main, BEST_MODEL_PATH

full_model, full_history = main(num_epochs=20, stride=2048)

full_last_epoch = len(full_history["train_loss"])

print(f"Train Loss: {full_history['train_loss'][-1]:.2f}")
print(f"Train Accuracy: {full_history['train_accuracy'][-1] * 100:.1f}%")
print(f"Validation Loss: {full_history['validation_loss'][-1]:.2f}")
print(f"Validation Accuracy: {full_history['validation_accuracy'][-1] * 100:.1f}%")
print()
print(f"Best Validation Accuracy: {full_history['best_validation_accuracy'] * 100:.1f}%")
print(f"Model saved to {BEST_MODEL_PATH}")

Epoch 1/20 — train_loss=0.3177 train_acc=0.8673 val_loss=0.1473 val_acc=0.9424
Epoch 2/20 — train_loss=0.2203 train_acc=0.9087 val_loss=0.1518 val_acc=0.9394
Epoch 3/20 — train_loss=0.1926 train_acc=0.9156 val_loss=0.1705 val_acc=0.9206
Epoch 4/20 — train_loss=0.1669 train_acc=0.9231 val_loss=0.0965 val_acc=0.9603
Epoch 5/20 — train_loss=0.1490 train_acc=0.9289 val_loss=0.0828 val_acc=0.9529
Epoch 6/20 — train_loss=0.1389 train_acc=0.9330 val_loss=0.0854 val_acc=0.9500
Epoch 7/20 — train_loss=0.1292 train_acc=0.9382 val_loss=0.0806 val_acc=0.9553
Epoch 8/20 — train_loss=0.1217 train_acc=0.9412 val_loss=0.0869 val_acc=0.9524
Epoch 9/20 — train_loss=0.1160 train_acc=0.9447 val_loss=0.0702 val_acc=0.9669
Epoch 10/20 — train_loss=0.1112 train_acc=0.9466 val_loss=0.0686 val_acc=0.9631
Epoch 11/20 — train_loss=0.1083 train_acc=0.9487 val_loss=0.0628 val_acc=0.9717
Epoch 12/20 — train_loss=0.1044 train_acc=0.9507 val_loss=0.0598 val_acc=0.9764
Epoch 13/20 — train_loss=0.1018 train_acc=0.9510 

In [5]:
from model_1d.evaluate import evaluate, BEST_MODEL_PATH

results = evaluate(BEST_MODEL_PATH, stride=2048)

print(f"Test Loss: {results['test_loss']:.2f}")
print(f"Test Accuracy: {results['test_accuracy'] * 100:.1f}%")
print(f"Precision: {results['macro_precision'] * 100:.1f}%")
print(f"Recall: {results['macro_recall'] * 100:.1f}%")
print(f"F1-score: {results['macro_f1'] * 100:.1f}%")
print()
print("Classification report saved to results/1d/classification_report.txt")
print("Confusion matrix saved to results/1d/confusion_matrix/confusion_matrix.png")
print("Evaluation metrics saved to results/1d/evaluation.json")

Test Loss: 0.11
Test Accuracy: 93.7%
Precision: 94.4%
Recall: 93.7%
F1-score: 93.6%

Classification report saved to results/1d/classification_report.txt
Confusion matrix saved to results/1d/confusion_matrix/confusion_matrix.png
Evaluation metrics saved to results/1d/evaluation.json


In [6]:
from model_1d.predict import predict_path,print_prediction

results = predict_path()

for index, result in enumerate(results):
    if index > 0:
        print()
    print_prediction(result)

File:
C:\Users\ishas\Downloads\ISSA\predict_samples\ADS_B.npy

Predicted Class:
ADS_B

Confidence:
98.8%

File:
C:\Users\ishas\Downloads\ISSA\predict_samples\FM_broadcast_20260126_023055_711669.npy

Predicted Class:
FM_broadcast

Confidence:
94.9%

File:
C:\Users\ishas\Downloads\ISSA\predict_samples\ism.npy

Predicted Class:
ISM_sensors

Confidence:
99.2%

File:
C:\Users\ishas\Downloads\ISSA\predict_samples\live.npy

Predicted Class:
FM_broadcast

Confidence:
100.0%
